# Vision Transformers: images as patches
**LLMs From Basics · V1.1 hands-on** &nbsp; [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abhishekaryan23/LLM-Learner/blob/main/course/notebooks/v1-1-vit-patches.ipynb)

In the lesson you watched an image turn into a row of patch tokens. Here you'll build that pipeline in PyTorch and train a tiny ViT:

1. **patchify** an image tensor into a sequence of flattened patches
2. **count tokens** and attention pairs for any resolution and patch size
3. a **patch embedding** with a [CLS] token and learned position embeddings, checked against a stride-$P$ convolution
4. a **tiny ViT** classifier, trained on a synthetic shapes dataset in well under a minute on CPU

**How to use it:** run cells top to bottom (Shift+Enter). Each `# TODO` cell is followed by a ✅ check cell. Stuck for more than five minutes? The **Solutions** are at the bottom.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
rng = np.random.default_rng(0)
S = 32                                           # image side in pixels
CLASSES = ["square", "circle", "triangle", "cross"]

def make_shapes(n, rng):
    # n noisy 3 x 32 x 32 images, each with one coloured shape at a random place and size.
    X = np.zeros((n, 3, S, S), np.float32)
    y = rng.integers(0, len(CLASSES), n)
    yy, xx = np.mgrid[0:S, 0:S]
    for i in range(n):
        r = rng.uniform(5, 9)
        cx, cy = rng.uniform(r + 1, S - r - 1, 2)
        k = y[i]
        if k == 0:   m = (abs(xx - cx) <= r * 0.8) & (abs(yy - cy) <= r * 0.8)
        elif k == 1: m = (xx - cx) ** 2 + (yy - cy) ** 2 <= r * r
        elif k == 2: m = (yy - cy <= r * 0.8) & (yy - cy >= -r) & (abs(xx - cx) <= (yy - cy + r) * 0.55)
        else:        m = ((abs(xx - cx) <= r) & (abs(yy - cy) <= r * 0.3)) | ((abs(yy - cy) <= r) & (abs(xx - cx) <= r * 0.3))
        X[i] = 0.1 * rng.standard_normal((3, S, S))
        X[i][:, m] += rng.uniform(0.4, 1.0, 3)[:, None]
    return torch.tensor(X), torch.tensor(y)

Xtr, ytr = make_shapes(2000, rng)
Xte, yte = make_shapes(600, rng)
print("train:", tuple(Xtr.shape), " test:", tuple(Xte.shape))

fig, axes = plt.subplots(1, 8, figsize=(12, 1.8))
for ax, img, lab in zip(axes, Xtr[:8], ytr[:8]):
    ax.imshow(img.permute(1, 2, 0).clamp(0, 1)); ax.set_title(CLASSES[lab]); ax.axis("off")
plt.show()

## 1. Patchify
`patchify(x, P)` takes a batch `x` of shape $(B, C, H, W)$ and returns $(B, N, P \cdot P \cdot C)$ with $N = (H/P)(W/P)$.

* Patches are ordered **row by row** (left to right, top to bottom), like reading a page.
* Inside a patch, flatten in the order (pixel row, pixel column, channel), as the original ViT code does.

Hint: `reshape` to $(B, C, H/P, P, W/P, P)$, `permute` the axes to $(B, H/P, W/P, P, P, C)$, then `reshape` again.

In [ ]:
def patchify(x, P):
    # TODO: return a (B, N, P*P*C) tensor of flattened patches, patches in row-major order.
    raise NotImplementedError("Fill in patchify")

In [ ]:
x = torch.arange(2 * 3 * 8 * 8, dtype=torch.float32).reshape(2, 3, 8, 8)
p = patchify(x, 4)
assert p.shape == (2, 4, 48), p.shape
# patch 1 is the top-right 4 x 4 block; flattened as (row, col, channel)
expected = x[0, :, 0:4, 4:8].permute(1, 2, 0).reshape(-1)
assert torch.equal(p[0, 1], expected), "Patch 1 should be the top-right block, flattened as (row, col, channel)."
assert torch.equal(p[1, 2], x[1, :, 4:8, 0:4].permute(1, 2, 0).reshape(-1)), "Patch 2 should be the bottom-left block."
print("✅ patchify works. A 224 x 224 RGB image with P = 16 gives", tuple(patchify(torch.zeros(1, 3, 224, 224), 16).shape))

## 2. Counting tokens
Write `n_tokens(H, W, P, cls=True)`, the sequence length a ViT sees (patches, plus one if there is a [CLS] token), and `attn_pairs(H, W, P)`, the number of query–key scores per head per layer, $n^2$, including [CLS].

In [ ]:
def n_tokens(H, W, P, cls=True):
    # TODO
    raise NotImplementedError("Fill in n_tokens")

def attn_pairs(H, W, P):
    # TODO
    raise NotImplementedError("Fill in attn_pairs")

In [ ]:
assert n_tokens(224, 224, 16) == 197
assert n_tokens(224, 224, 16, cls=False) == 196
assert n_tokens(224, 224, 14, cls=False) == 256       # ViT-L/14
assert n_tokens(448, 448, 16) == 785
assert attn_pairs(224, 224, 16) == 197 ** 2 == 38809
ratio = attn_pairs(448, 448, 16) / attn_pairs(224, 224, 16)
assert abs(ratio - 15.88) < 0.01, ratio
print(f"✅ Doubling the side: {n_tokens(448, 448, 16) / n_tokens(224, 224, 16):.2f}x tokens, {ratio:.2f}x attention pairs.")

## 3. Patch embedding, [CLS] and positions
$$z_0 = \big[\,x_{\text{cls}};\; x_p^1 E;\; \dots;\; x_p^N E\,\big] + E_{\text{pos}}$$

Complete `PatchEmbed.forward`:
1. patchify, then apply the linear layer `self.proj` (this is $E$, plus a bias),
2. prepend `self.cls`, expanded to the batch size,
3. add `self.pos`.

The output has shape $(B, N+1, D)$.

In [ ]:
class PatchEmbed(nn.Module):
    def __init__(self, img=32, P=8, C=3, D=64):
        super().__init__()
        self.P = P
        N = (img // P) ** 2
        self.proj = nn.Linear(P * P * C, D)                     # E (and a bias)
        self.cls = nn.Parameter(torch.zeros(1, 1, D))           # learned [CLS] token
        self.pos = nn.Parameter(torch.randn(1, N + 1, D) * 0.02)  # learned position embeddings

    def forward(self, x):
        # TODO: patchify -> project -> prepend cls -> add positions
        raise NotImplementedError("Fill in PatchEmbed.forward")

In [ ]:
pe = PatchEmbed(img=32, P=8, C=3, D=64)
z = pe(Xtr[:5])
assert z.shape == (5, 17, 64), z.shape
assert torch.allclose(z[:, 0], (pe.cls[0, 0] + pe.pos[0, 0]).expand(5, -1)), "Token 0 should be cls + pos[0]."

# The patch embedding is exactly a convolution with kernel = stride = P.
conv = nn.Conv2d(3, 64, kernel_size=8, stride=8)
with torch.no_grad():
    conv.weight.copy_(pe.proj.weight.view(64, 8, 8, 3).permute(0, 3, 1, 2))
    conv.bias.copy_(pe.proj.bias)
    via_conv = conv(Xtr[:5]).flatten(2).transpose(1, 2)        # (B, N, D), row-major
    assert torch.allclose(via_conv + pe.pos[:, 1:], z[:, 1:], atol=1e-5), "Patch tokens should match the stride-8 convolution."
print("✅ PatchEmbed works, and equals a stride-P convolution:", tuple(z.shape))

## 4. A tiny ViT
The encoder is PyTorch's standard `nn.TransformerEncoder` with pre-norm blocks, the same block you met in M1.4, minus the causal mask. Complete `forward`: embed, encode, take the **[CLS]** output (token 0), apply the final `LayerNorm` and the linear head.

In [ ]:
class TinyViT(nn.Module):
    def __init__(self, img=32, P=8, D=64, depth=2, heads=4, n_classes=4):
        super().__init__()
        self.embed = PatchEmbed(img, P, 3, D)
        layer = nn.TransformerEncoderLayer(D, heads, 4 * D, dropout=0.0, activation="gelu",
                                           batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, depth, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(D)
        self.head = nn.Linear(D, n_classes)

    def forward(self, x):
        # TODO: embed -> encoder -> take token 0 -> norm -> head
        raise NotImplementedError("Fill in TinyViT.forward")

In [ ]:
def train(model, epochs=12, bs=64, lr=3e-3):
    opt = torch.optim.AdamW(model.parameters(), lr, weight_decay=0.05)
    steps = epochs * ((len(Xtr) + bs - 1) // bs)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, lr, total_steps=steps)
    for ep in range(epochs):
        perm = torch.randperm(len(Xtr))
        for i in range(0, len(Xtr), bs):
            idx = perm[i:i + bs]
            loss = F.cross_entropy(model(Xtr[idx]), ytr[idx])
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    return model

def accuracy(model, X, y):
    with torch.no_grad():
        return (model(X).argmax(1) == y).float().mean().item()

torch.manual_seed(0)
t0 = time.time()
vit = train(TinyViT())
acc = accuracy(vit, Xte, yte)
print(f"test accuracy {acc:.3f} after {time.time() - t0:.1f}s, {sum(p.numel() for p in vit.parameters()):,} parameters")
assert acc > 0.9, "Expected > 90% test accuracy. Check TinyViT.forward."
print("✅ Your tiny ViT learned to recognise shapes from 16 patch tokens.")

## Experiment
1. **Shuffle the patches.** At test time, permute the 16 patch tokens *after* the position embeddings are added, so each patch keeps its own position. Then permute them *before*, so patches land on the wrong positions. Predict first: which one hurts? (On this toy task you may be surprised: a circle or a cross can often be recognised from its pieces alone, a "bag of patches". Can you design a class pair that only positions can tell apart, say a shape in the left half vs the right half?)
2. **Look at the position embeddings.** The cell below plots the cosine similarity between each learned position embedding and the others, arranged on the 4 × 4 grid. Do nearby positions end up similar, as in the ViT paper?
3. **Change the patch size.** Train `TinyViT(P=4)` (64 tokens) and compare time and accuracy with `P=8` (16 tokens) and `P=16` (4 tokens).

In [ ]:
# 1. shuffling patches before vs after the position embeddings
perm = torch.randperm(16) + 1                       # patch tokens are 1..16; token 0 is [CLS]
order = torch.cat([torch.tensor([0]), perm])
with torch.no_grad():
    pe = vit.embed
    t = pe.proj(patchify(Xte, pe.P))
    t = torch.cat([pe.cls.expand(len(Xte), -1, -1), t], 1)
    after = (t + pe.pos)[:, order]                  # each patch keeps its own position
    before = t[:, order] + pe.pos                   # patches get the wrong positions
    for name, z in [("shuffled after pos", after), ("shuffled before pos", before)]:
        out = vit.head(vit.norm(vit.encoder(z)[:, 0]))
        print(f"{name:>20}: accuracy {(out.argmax(1) == yte).float().mean().item():.3f}")

# 2. similarity of learned position embeddings
P = F.normalize(vit.embed.pos[0, 1:].detach(), dim=1)
sim = (P @ P.T).reshape(16, 4, 4)
fig, axes = plt.subplots(1, 4, figsize=(9, 2.4))
for ax, k in zip(axes, [0, 5, 10, 15]):
    ax.imshow(sim[k], cmap="viridis"); ax.set_title(f"vs position {k}"); ax.axis("off")
plt.show()

---
## Solutions
Try the exercises first. Each solution below can be pasted into its TODO cell.

In [ ]:
# Solution: patchify
def patchify(x, P):
    B, C, H, W = x.shape
    x = x.reshape(B, C, H // P, P, W // P, P)
    x = x.permute(0, 2, 4, 3, 5, 1)                  # B, H/P, W/P, P, P, C
    return x.reshape(B, (H // P) * (W // P), P * P * C)

In [ ]:
# Solution: counting tokens
def n_tokens(H, W, P, cls=True):
    return (H // P) * (W // P) + (1 if cls else 0)

def attn_pairs(H, W, P):
    return n_tokens(H, W, P) ** 2

In [ ]:
# Solution: PatchEmbed
class PatchEmbed(nn.Module):
    def __init__(self, img=32, P=8, C=3, D=64):
        super().__init__()
        self.P = P
        N = (img // P) ** 2
        self.proj = nn.Linear(P * P * C, D)
        self.cls = nn.Parameter(torch.zeros(1, 1, D))
        self.pos = nn.Parameter(torch.randn(1, N + 1, D) * 0.02)

    def forward(self, x):
        t = self.proj(patchify(x, self.P))                    # (B, N, D)
        cls = self.cls.expand(x.shape[0], -1, -1)             # (B, 1, D)
        return torch.cat([cls, t], dim=1) + self.pos          # (B, N+1, D)

In [ ]:
# Solution: TinyViT
class TinyViT(nn.Module):
    def __init__(self, img=32, P=8, D=64, depth=2, heads=4, n_classes=4):
        super().__init__()
        self.embed = PatchEmbed(img, P, 3, D)
        layer = nn.TransformerEncoderLayer(D, heads, 4 * D, dropout=0.0, activation="gelu",
                                           batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, depth, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(D)
        self.head = nn.Linear(D, n_classes)

    def forward(self, x):
        z = self.encoder(self.embed(x))
        return self.head(self.norm(z[:, 0]))